# Laya fine-tune v3: 30-minute horizon

v2 trained on **5-min** labels and lost to the majority class (OVERALL 0.6168 vs 0.6989) — not because the model failed to learn (+28.5pp acc, Brier 0.787 -> 0.436) but because the 5-min forward direction is a coin flip: 48.7% up, breakeven win rate 68.4% at the 12.7bps hurdle.

v3 retrains on the **30-min** horizon (`kinan21/laya-hl-trial-v3-h30`), where the bar is far lower: **breakeven win rate 60.0%**, median |fwd| 30.15bps vs the same 12.7bps hurdle.

**This notebook trains AND backtests in one run**, so the checkpoint never has to round-trip through a download+reupload to score it. Trade math is byte-identical to `scripts/backtest.py:236-260`.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("cuda:", torch.cuda.is_available(), "| gpus:", torch.cuda.device_count())


## 2. Install deps

In [ ]:
!pip install -q -U "laya>=0.3.21" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scikit-learn


## 3. Download model + preprocess our data

In [ ]:
import os, json, torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya-multilingual"

# Dataset attached as Kaggle input: kinan21/laya-hl-trial-v3-h30 (30-min horizon labels)
DATA_DIR = None
for cand in ["/kaggle/input/laya-hl-trial-v3-h30",
             "/kaggle/input/datasets/kinan21/laya-hl-trial-v3-h30"]:
    if os.path.exists(os.path.join(cand, "train.jsonl")):
        DATA_DIR = cand
        break
assert DATA_DIR, "Dataset kinan21/laya-hl-trial-v3-h30 not mounted. Attach it via Add Input."
print("DATA_DIR =", DATA_DIR)
print(f"Fetching model from {MODEL_ID} (~1 min on Kaggle's network)...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label}

def load_rows(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

for split in ["train", "val"]:
    rows = load_rows(os.path.join(DATA_DIR, f"{split}.jsonl"))
    items, dropped = [], 0
    for row in rows:
        # NOTE: our state is already rendered text (laya-live prompt), NOT a JSON dict
        state = row["state"] if isinstance(row["state"], str) else json.dumps(row["state"])
        questions = json.loads(row["questions"]) if isinstance(row["questions"], str) else row["questions"]
        gold = json.loads(row["gold"]) if isinstance(row["gold"], str) else row["gold"]
        for qid, q in questions.items():
            if qid in gold:
                it = build_training_item(state, q, gold[qid])
                if it:
                    items.append(it)
                else:
                    dropped += 1
    torch.save(items, f"/kaggle/working/{split}_items.pt")
    print(f"{split}: {len(rows)} cases -> {len(items)} items (dropped {dropped})")

with open("/kaggle/working/model_dir.txt", "w") as f:
    f.write(model_dir)
print("model_dir:", model_dir)



## 4. Training script (official Laya RLCD, unmodified)

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    
    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)

    # Hold the calibration slice out of training before sharding. Temperatures fitted on
    # items the run has already trained on measure the fit rather than the calibration: the
    # model is near-certain and near-correct on them, so the optimiser has nothing to soften
    # and returns a degenerate scale. The seed is fixed and rank-independent, so every rank
    # withholds exactly the same items and none of them reaches a training batch.
    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)
    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]
    # Even split with drop_last-style padding: a rank that runs one optimizer
    # step more than the others hangs the DDP all-reduce at the end of the
    # epoch (the partner is already in the barrier). Pads to an exact multiple
    # of the per-rank effective batch so every rank steps in lockstep.
    per_rank = math.ceil(len(train_items) / world_size)
    padded = train_items + train_items[:per_rank * world_size - len(train_items)]
    my_items = padded[rank * per_rank:(rank + 1) * per_rank]
    
    EPOCHS = 4
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # GRPO baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    # Same value on every rank: padding made the per-rank counts identical,
    # so the cosine schedule lines up with the real number of optimizer steps.
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        # Reproducible sharding: shuffle once on all ranks with a rank-independent
        # seed, then shard. Per-rank seeds would give each rank a different item
        # order -- both ranks still cover the data, but no two ranks agree on what
        # a batch contains, and any epoch bookkeeping that assumes rank-coherent
        # batches is silently wrong.
        order = list(range(len(train_items)))
        random.Random(20260922 + epoch).shuffle(order)
        train_items = [train_items[i] for i in order]

        # Even split with drop_last-style padding: a rank that runs one optimizer
        # step more than the others hangs the DDP all-reduce at the end of the
        # epoch (the partner is already in the barrier). Pads to an exact multiple
        # of the per-rank effective batch so every rank steps in lockstep.
        per_rank = math.ceil(len(train_items) / world_size)
        padded = train_items + train_items[:per_rank * world_size - len(train_items)]
        my_items = padded[rank * per_rank:(rank + 1) * per_rank]

        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            # 1. Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            # 2. Evaluate proper scoring reward (w_sph=0.75 for soft target matching)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            # 3. Policy gradient loss + full 1.0 soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()
    
    # Post-training temperature calibration on rank 0 (micro-batched in chunks of 16 to prevent OOM)
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        # calib_items was held out above and never entered a training batch
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))
        
        fitted_temps = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-typed-decisions"
        cfg["temperature"] = fitted_temps
        # This fit is per type; inherited bucket overrides would hide the new values.
        cfg.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()



## 5. Launch training

In [ ]:
import os, subprocess, torch
n_gpu = torch.cuda.device_count()
print(f"GPUs available: {n_gpu}")
model_dir = open("/kaggle/working/model_dir.txt").read().strip()
OUTPUT_DIR = "/kaggle/working/laya_hl_small_h30"
os.makedirs(OUTPUT_DIR, exist_ok=True)
nproc = max(1, n_gpu)
cmd = (
    f"torchrun --standalone --nproc_per_node={nproc} /kaggle/working/train_ddp.py "
    f"{model_dir} {OUTPUT_DIR}"
)
print("Launching:", cmd)
ret = subprocess.run(cmd, shell=True)
print("torchrun exit code:", ret.returncode)
assert ret.returncode == 0, "training failed"



## 6. Eval: baseline vs finetuned on our val set

In [ ]:
import time, json, os, statistics
from collections import Counter, defaultdict
import numpy as np
import laya

DATA_DIR = None
for cand in ["/kaggle/input/laya-hl-trial-v3-h30",
             "/kaggle/input/datasets/kinan21/laya-hl-trial-v3-h30"]:
    if os.path.exists(os.path.join(cand, "val.jsonl")):
        DATA_DIR = cand
        break
assert DATA_DIR, "dataset not mounted"
model_dir = open("/kaggle/working/model_dir.txt").read().strip()
OUTPUT_DIR = "/kaggle/working/laya_hl_small_h30"

print("DATA_DIR =", DATA_DIR)
print("model_dir =", model_dir)
print("OUTPUT_DIR =", OUTPUT_DIR)

def load_rows(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

val_rows = load_rows(os.path.join(DATA_DIR, "val.jsonl"))
tr_rows  = load_rows(os.path.join(DATA_DIR, "train.jsonl"))
print(f"train cases: {len(tr_rows)}  val cases: {len(val_rows)}")

# ---- scoring constants (byte-identical to scripts/backtest.py) ----
TAKER_FEE_BPS = 0.35
ADVERSE_SELECTION_BPS = 12.0

def gold_label(gold_q, keys):
    """The dataset stores soft probabilities, not a hard label. Derive the argmax.
    v2 crashed here reading gold[qid]['label'], which does not exist."""
    p = gold_q["probabilities"]
    return max(keys, key=lambda k: p.get(k, 0.0))

def evaluate(agent, rows, tag, want_pnl=False, limit=None):
    if limit:
        rows = rows[:limit]
    acc, brier, lat = {}, {}, []
    trades, wins, net_bps = 0, 0, 0.0
    pnl = []
    for i, row in enumerate(rows):
        state = row["state"] if isinstance(row["state"], str) else json.dumps(row["state"])
        questions = row["questions"] if isinstance(row.get("questions"), dict) else json.loads(row["questions"])
        gold = row["gold"] if isinstance(row.get("gold"), dict) else json.loads(row["gold"])
        t0 = time.perf_counter()
        res = agent.predict(state, questions)
        lat.append((time.perf_counter() - t0) * 1000)
        ans = res["answers"]
        for qid, qdef in questions.items():
            if qid not in gold:
                continue
            keys = list(qdef["criteria"].keys())
            pred_choice = ans[qid]["choice"]
            lab = gold_label(gold[qid], keys)
            acc.setdefault(qid, []).append(float(pred_choice == lab))
            p = np.array([ans[qid]["probabilities"].get(k, 1e-6) for k in keys], dtype=np.float64)
            p = p / p.sum()
            g = np.zeros(len(keys)); g[keys.index(lab)] = 1.0
            brier.setdefault(qid, []).append(float(((p - g) ** 2).sum()))
        # ---- trade decision (backtest.py semantics) ----
        intent_pred = ans.get("intent", {}).get("choice", "hold")
        if intent_pred == "open":
            bias_pred = ans.get("bias", {}).get("choice", "long")
            direction = 1.0 if bias_pred == "long" else -1.0
            fwd = row["meta"]["fwd_bps"]
            hurdle = max(row["meta"].get("hurdle_bps") or 0.0,
                         ADVERSE_SELECTION_BPS + 2 * TAKER_FEE_BPS)
            tp = direction * fwd - hurdle
            trades += 1; net_bps += tp
            if tp > 0:
                wins += 1
            if want_pnl:
                pnl.append({**{k: row["meta"][k] for k in ("coin", "ts", "day")},
                            "direction": direction, "fwd_bps": fwd,
                            "hurdle_bps": hurdle, "trade_pnl_bps": tp})
        if i % 500 == 0:
            print(f"  ..{i}/{len(rows)}", flush=True)
    out = {qid: float(np.mean(v)) for qid, v in acc.items()}
    out["overall_acc"] = float(np.mean([v for vs in acc.values() for v in vs]))
    out["overall_brier"] = float(np.mean([v for vs in brier.values() for v in vs]))
    out["mean_latency_ms"] = float(np.mean(lat))
    out["trades"] = trades
    out["net_bps"] = net_bps
    out["mean_trade_bps"] = (net_bps / trades) if trades else 0.0
    out["win_rate"] = (wins / trades) if trades else 0.0
    print(f"[{tag}] " + " | ".join(f"{k}={v:.4f}" for k, v in out.items() if k not in ("trades", "net_bps"))
          + f" | trades={trades} net={net_bps:.1f}", flush=True)
    return (out, pnl) if want_pnl else out

def majority_ref(rows, split_name):
    """Majority-class reference on the SAME rows, so we never confuse
    'the model beat nothing' with 'the model beat something'."""
    per = {}
    for r in rows:
        for qid, qdef in r["questions"].items():
            if qid not in r["gold"]:
                continue
            keys = list(qdef["criteria"].keys())
            per.setdefault(qid, []).append(gold_label(r["gold"][qid], keys))
    out = {q: Counter(v).most_common(1)[1] / len(v) if False else Counter(v).most_common(1)[0][1] / len(v)
           for q, v in per.items()}
    out["overall_acc"] = float(np.mean(list(out.values())))
    cost = ADVERSE_SELECTION_BPS + 2 * TAKER_FEE_BPS
    out["always_long_mean_bps"] = float(np.mean([r["meta"]["fwd_bps"] - cost for r in rows]))
    out["n"] = len(rows)
    print(f"[{split_name} MAJORITY] " + " | ".join(f"{k}={v:.4f}" for k, v in out.items()))
    return out

print("\n== MAJORITY-CLASS REFERENCE ==")
maj_val = majority_ref(val_rows, "VAL")
maj_tr  = majority_ref(tr_rows,  "TRAIN")

print("\n== BASELINE (pretrained laya-multilingual) ==")
agent_base = laya.Agent(model_dir, device="cuda")
m_base_tr = evaluate(agent_base, tr_rows, "base/train")
m_base, pnl_base = evaluate(agent_base, val_rows, "base/val", want_pnl=True)
del agent_base
import torch; torch.cuda.empty_cache()

print("\n== FINETUNED (30-min) ==")
agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")
m_ft_tr = evaluate(agent_ft, tr_rows, "ft/train")
m_ft, pnl_ft = evaluate(agent_ft, val_rows, "ft/val", want_pnl=True)
del agent_ft
torch.cuda.empty_cache()

# ----------------------------- report -----------------------------
report = {
    "baseline": m_base, "finetuned": m_ft,
    "baseline_train": m_base_tr, "finetuned_train": m_ft_tr,
    "majority_val": maj_val, "majority_train": maj_tr,
    "val_cases": len(val_rows),
    "horizon_min": 30,
    "breakeven_note": "win rate needed at a 12.7bps hurdle: 68.4% (5m) / 63.1% (15m) / 60.0% (30m); "
                      "this dataset is 30-min labelled, so the bar is 60.0%",
    "note": "v2 (5-min) scored ft OVERALL 0.6168 vs majority 0.6989, -12.03 bps/trade, win 32.5%",
}
with open("/kaggle/working/eval_report.json", "w") as f:
    json.dump(report, f, indent=2)
with open("/kaggle/working/backtest_pnl_ft.jsonl", "w") as f:
    for t in pnl_ft:
        f.write(json.dumps(t) + "\n")
with open("/kaggle/working/backtest_pnl_base.jsonl", "w") as f:
    for t in pnl_base:
        f.write(json.dumps(t) + "\n")

print("\n" + "=" * 78)
print("HEAD-TO-HEAD (30-min horizon, val = held-out)")
print("=" * 78)
rows_show = [
    ("bias acc",     m_base["bias"],        m_ft["bias"],        maj_val["bias"]),
    ("intent acc",   m_base["intent"],      m_ft["intent"],      maj_val["intent"]),
    ("leverage acc", m_base["leverage"],    m_ft["leverage"],    maj_val["leverage"]),
    ("OVERALL acc",  m_base["overall_acc"], m_ft["overall_acc"], maj_val["overall_acc"]),
]
print(f"{'metric':16} {'base':>9} {'finetuned':>11} {'majority':>9} {'ft-base':>9}")
for name, b, f_, m in rows_show:
    print(f"{name:16} {b:9.4f} {f_:11.4f} {m:9.4f} {f_-b:+9.4f}")
print(f"{'brier':16} {m_base['overall_brier']:9.4f} {m_ft['overall_brier']:11.4f} {'-':>9}")
print()
print(f"{'trades':16} {m_base['trades']:9d} {m_ft['trades']:11d}")
print(f"{'net bps':16} {m_base['net_bps']:9.1f} {m_ft['net_bps']:11.1f}")
print(f"{'mean bps/trade':16} {m_base['mean_trade_bps']:9.2f} {m_ft['mean_trade_bps']:11.2f}"
      f"   (always-long ref {maj_val['always_long_mean_bps']:+.2f})")
print(f"{'win rate':16} {m_base['win_rate']:9.4f} {m_ft['win_rate']:11.4f}"
      f"   <- breakeven 60.0% at 30min")
print(f"{'latency ms':16} {m_base['mean_latency_ms']:9.0f} {m_ft['mean_latency_ms']:11.0f}")

print("\nper-coin PnL (finetuned):")
by_coin = defaultdict(lambda: [0, 0.0, 0])
for t in pnl_ft:
    c = by_coin[t["coin"]]; c[0] += 1; c[1] += t["trade_pnl_bps"]
    if t["trade_pnl_bps"] > 0:
        c[2] += 1
print(f"{'Coin':6} {'Trades':>7} {'Net(bps)':>10} {'WR%':>6}")
for coin in sorted(by_coin):
    n, net, w = by_coin[coin]
    print(f"{coin:6} {n:7d} {net:10.1f} {100*w/n:6.1f}")

print("\nsaved /kaggle/working/eval_report.json + backtest_pnl_{ft,base}.jsonl")
